In [ ]:
import numpy as np
import random
import seaborn as sns
import matplotlib.pyplot as plt

import os
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, random_split
from sklearn.metrics import classification_report, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
seed = 42

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Data Loading

In [ ]:
# Load Datasets
def load_images2(class_folder, img_size=(512, 512), limit=100):
    count = 0
    X = []

    for file in os.listdir(class_folder):
        if count >= limit:
            break
        img_path = os.path.join(class_folder, file)

        # Read image (BGR)
        img = cv2.imread(img_path) 

        if img is None:
            continue  # skip corrupted files

        img = cv2.resize(img, img_size) # Resizing led to worse quality images with fewer keypoints

        X.append(img)
        count += 1

    return np.array(X)

In [ ]:
train_folder = "Datasets/road/train"
test_folder = "Datasets/road/test"

train_img = []
train_label = []

image_files = sorted([f for f in os.listdir(train_folder) if f.endswith('_sat.jpg')])
mask_files = sorted([f for f in os.listdir(train_folder) if f.endswith('_mask.png')])

LIMIT = 10  
IMG_SIZE = (512, 512)

for img_file, mask_file in zip(image_files[:LIMIT], mask_files[:LIMIT]):
    img = cv2.imread(os.path.join(train_folder, img_file))
    mask = cv2.imread(os.path.join(train_folder, mask_file), cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        continue
    
    img = cv2.resize(img, IMG_SIZE)
    mask = cv2.resize(mask, IMG_SIZE, interpolation=cv2.INTER_NEAREST)
    binary_mask = (mask > 127).astype(np.uint8)

    train_img.append(img)
    train_label.append(binary_mask)

In [ ]:
X_train = np.array(train_img)
Y_train = np.array(train_label)
X_test = load_images2("Datasets/road/test")

print("X train shape: ",X_train.shape)
print("Y Train shape: ",Y_train.shape)
print("X Test shape: ",X_test.shape)

In [ ]:
# Normalizing pixel values to prevent exploding gradients
X_train = X_train / 255.0
X_test = X_test / 255.0

## Dataset Preparation

- Data converted to PyTorch tensors
- Train-validation split applied (80-20)
- DataLoader used for batching and shuffling

In [ ]:
# Convert to Pytorch Format

X_train_torch = torch.tensor(X_train, dtype=torch.float32).permute(0, 3, 1, 2)
X_test_torch = torch.tensor(X_test, dtype=torch.float32).permute(0, 3, 1, 2)

Y_train_torch = torch.tensor(Y_train, dtype=torch.long)

print("Train X shape:", X_train_torch.shape)
print("Train Y shape:", Y_train_torch.shape)
print("Test X shape:", X_test_torch.shape)

In [ ]:
# Create DataLoader

train_dataset = TensorDataset(X_train_torch, Y_train_torch)

train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

train_dataset, val_dataset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

## Model Architecture

In [ ]:
def double_conv(in_c, out_c):
    return nn.Sequential(
        nn.Conv2d(in_c, out_c, kernel_size=3, padding=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_c, out_c, kernel_size=3, padding=1),
        nn.ReLU(inplace=True)
    )

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super(UNet, self).__init__()

        # Encoder (Downsampling)
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.pool = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = double_conv(128, 256)
        
        # Decoder (Upsampling)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = double_conv(256, 128) # 128 (up) + 128 (skip) = 256
        
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = double_conv(128, 64) # 64 (up) + 64 (skip) = 128
        
        # Output layer
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)


    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        
        # Bottleneck
        b = self.bottleneck(self.pool(e2))
        
        # Decoder with Skip Connections
        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1) # Concatenate along channel dim
        d2 = self.dec2(d2)
        
        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        
        return self.final_conv(d1)

# Training

In [ ]:
# Training setup

model = UNet()

#criterion = nn.CrossEntropyLoss()
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(Y_train.flatten()),
    y=Y_train.flatten()
)

class_weights = torch.tensor(class_weights, dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Training Loop

epochs = 10
train_accs = []
val_accs = []

for epoch in range(epochs):

    # TRAINING
    model.train()   # sets model to training mode -> affects dropout(enabled), BatchNorm
    correct = 0
    total = 0
    running_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()   # clears old gradients

        outputs = model(images) # Prediction/Forward prop
        loss = criterion(outputs, labels) # Loss computed

        loss.backward()     # Gradients computed
        optimizer.step()    # Gradients updated

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = correct / total

    # VALIDATION 
    model.eval()        # Sets model to evaluation mode -> dropout(disabled), BatchNorm uses previously stored mean and variance instead of computing a new one
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")